# Telco Customer Churn — Makine Öğrenmesi Projesi

Bu notebook, veri kaynağını, keşifsel analizi, veri hazırlığını ve `src/train.py` tarafından eğitilip kaydedilen modellerin gerçek test sonuçlarını inceler. Model eğitim kodunun asıl ve tekrar üretilebilir kaynağı `src/train.py` dosyasıdır.

**Problem:** Müşteri aboneliğini bırakacak mı? Hedef `Churn` (Yes/No). Bu çalışma karar destek amaçlı eğitim projesidir.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import joblib

ROOT = Path.cwd().parent
DATA_PATH = ROOT / "data" / "raw" / "Telco-Customer-Churn.csv"
OUT = ROOT / "outputs"
df_raw = pd.read_csv(DATA_PATH)
print("Ham veri boyutu:", df_raw.shape)
print("Sütun sayısı:", len(df_raw.columns))
print("Eksik değer bulunan sütunlar:")
print(df_raw.isna().sum().loc[lambda s: s.gt(0)])
display(df_raw.head())

Ham veri boyutu: (7043, 21)
Sütun sayısı: 21
Eksik değer bulunan sütunlar:
Series([], dtype: int64)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1. Veri keşfi ve görselleştirme
En az üç inceleme: hedef sınıf dağılımı, abonelik süresi ile churn ilişkisi ve sözleşme türüne göre churn.

In [2]:
fig, ax = plt.subplots(figsize=(6,4))
df_raw["Churn"].value_counts().reindex(["No","Yes"]).plot(kind="bar", ax=ax)
ax.set_title("Müşteri Kaybı Dağılımı"); ax.set_xlabel("Churn"); ax.set_ylabel("Müşteri sayısı"); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7,4))
df_raw.boxplot(column="tenure", by="Churn", ax=ax)
ax.set_title("Churn Durumuna Göre Abonelik Süresi"); ax.set_xlabel("Churn"); ax.set_ylabel("Ay"); plt.suptitle(""); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8,4))
pd.crosstab(df_raw["Contract"], df_raw["Churn"]).reindex(columns=["No","Yes"]).plot(kind="bar", ax=ax)
ax.set_title("Sözleşme Türüne Göre Churn"); ax.set_xlabel("Sözleşme türü"); ax.set_ylabel("Müşteri sayısı"); plt.xticks(rotation=15); plt.tight_layout(); plt.show()

/tmp/ipykernel_1842/593440606.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_title("Müşteri Kaybı Dağılımı"); ax.set_xlabel("Churn"); ax.set_ylabel("Müşteri sayısı"); plt.tight_layout(); plt.show()
/tmp/ipykernel_1842/593440606.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_title("Churn Durumuna Göre Abonelik Süresi"); ax.set_xlabel("Churn"); ax.set_ylabel("Ay"); plt.suptitle(""); plt.tight_layout(); plt.show()
/tmp/ipykernel_1842/593440606.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_title("Sözleşme Türüne Göre Churn"); ax.set_xlabel("Sözleşme türü"); ax.set_ylabel("Müşteri sayısı"); plt.xticks(rotation=15); plt.tight_layout(); plt.show()


## 2. Temizleme ve model değerlendirmesi
`src/train.py` içindeki eğitim hattı `customerID` sütununu çıkarır, `TotalCharges` alanını sayısala çevirir, 11 eksik kaydı temizler, stratified %80/%20 bölme yapar ve kategorik/sayısal ön işlemeyi Pipeline içinde uygular.

In [3]:
metrics = pd.read_csv(OUT / "model_karsilastirma.csv")
print("Temiz veri satırı:", 7032)
print("Sınıf dağılımı (temiz veri):")
print(pd.Series({"Kalan":5163, "Ayrılan":1869}))
display(metrics.round(3))
best = metrics.sort_values("F1", ascending=False).iloc[0]
print("F1 ölçütüne göre seçilen model:", best["Model"])
print("Kaydedilmiş model dosyası mevcut:", (OUT / "en_iyi_model.joblib").exists())
model = joblib.load(OUT / "en_iyi_model.joblib")
print("Yüklenen model türü:", type(model).__name__)

Temiz veri satırı: 7032
Sınıf dağılımı (temiz veri):
Kalan      5163
Ayrılan    1869
dtype: int64


,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Random Forest,0.768,0.548,0.717,0.621,0.831
1,Logistic Regression,0.726,0.490,0.797,0.607,0.835


F1 ölçütüne göre seçilen model: Random Forest
Kaydedilmiş model dosyası mevcut: True


Yüklenen model türü: Pipeline


## 3. Hata analizi ve yorumlama
Random Forest F1 açısından önde; Logistic Regression churn sınıfında daha yüksek recall üretmiştir. İş hedefi kayıp müşterileri mümkün olduğunca kaçırmamaksa recall farkı dikkate alınmalıdır.

In [4]:
errors = pd.read_csv(OUT / "yanlis_tahminler.csv")
preds = pd.read_csv(OUT / "test_tahminleri.csv")
print("Test örneği:", len(preds))
print("Yanlış sınıflandırılan kayıt:", len(errors))
print("Hata oranı:", round(len(errors)/len(preds), 3))
display(errors.head(10))

Test örneği: 1407
Yanlış sınıflandırılan kayıt: 327
Hata oranı: 0.232


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Gerçek,Tahmin
0,Female,0,No,No,7,Yes,Yes,Fiber optic,No,Yes,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),78.55,522.95,0,1
1,Female,0,No,No,2,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.65,38.70,1,0
2,Male,1,Yes,No,4,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.20,280.35,0,1
3,Female,0,Yes,No,14,Yes,No,Fiber optic,No,No,No,No,No,Yes,Month-to-month,Yes,Electronic check,78.10,1122.40,0,1
4,Female,0,Yes,Yes,30,No,No phone service,DSL,No,No,No,Yes,Yes,Yes,Month-to-month,No,Credit card (automatic),51.20,1561.50,1,0
5,Female,0,No,No,4,Yes,Yes,DSL,No,No,No,Yes,Yes,No,Month-to-month,Yes,Mailed check,64.40,253.00,0,1
6,Male,0,No,No,1,Yes,No,DSL,No,No,No,Yes,No,No,Month-to-month,Yes,Electronic check,49.30,49.30,0,1
7,Female,0,Yes,Yes,48,Yes,Yes,Fiber optic,Yes,Yes,No,No,Yes,Yes,One year,Yes,Credit card (automatic),103.25,5037.55,1,0
8,Male,1,Yes,No,62,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Electronic check,103.75,6383.35,1,0
9,Male,0,No,No,1,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Mailed check,19.75,19.75,0,1


## 4. Sonuç ve sınırlılıklar

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| Random Forest | 0.768 | 0.548 | 0.717 | 0.621 | 0.831 |
| Logistic Regression | 0.726 | 0.490 | 0.797 | 0.607 | 0.835 |

Veri seti benchmark niteliğindedir. Zaman bazlı test, çapraz doğrulama, eşik optimizasyonu ve kalibrasyon sonraki geliştirmelerdir. Model kesin karar aracı değildir.